# A/B Test Walkthrough: Customer Onboarding Experiment

This notebook walks through the `ab_platform` package interactively: simulate data, run diagnostics, then frequentist, Bayesian, CUPED, and sequential-testing analyses.

Run `pip install -e ..` (or `pip install -e '.[dev]'`) from the repo root first.

In [ ]:
from ab_platform.config import ExperimentConfig
from ab_platform import simulate, diagnostics, frequentist, bayesian, cuped, sequential, viz

cfg = ExperimentConfig(csv_path='../data/onboarding_events.csv', db_path='../data/onboarding_ab.db', reports_dir='../reports')
df = simulate.run(cfg)
df.head()

## 1. Data quality gates (run these BEFORE trusting any result)

In [ ]:
diag = diagnostics.run_all_checks(df)
for k, v in diag.items():
    print(k, '->', v.get('verdict', v))

## 2. Primary metric: two-proportion z-test

In [ ]:
primary = frequentist.two_proportion_ztest(df)
primary

## 3. Segment consistency checks (Simpson's paradox guard)

In [ ]:
frequentist.segment_check(df, 'signup_channel')

## 4. CUPED variance reduction

In [ ]:
cuped_result = cuped.cuped_ttest(df)
cuped_result

## 5. Bayesian analysis

In [ ]:
bayes_result = bayesian.bayesian_ab_test(df)
bayes_rec = bayesian.bayesian_recommendation(bayes_result)
print(bayes_result['prob_b_beats_a'], bayes_rec['decision'])
viz.plot_bayesian_posteriors(bayes_result['posterior_a'], bayes_result['posterior_b'], '../reports/nb_bayes.png')

## 6. Sequential testing: why peeking is dangerous

In [ ]:
fpr = sequential.naive_peeking_false_positive_rate(true_effect_pp=0, n_peeks=10, n_per_arm=2000, baseline_rate=0.34, n_sims=500)
print(f'False positive rate with 10 naive peeks under the null: {fpr:.1%} (nominal alpha is 5%)')

## 7. Full pipeline via the `report` module

In [ ]:
from ab_platform import report
results = report.run_full_analysis(cfg)
report.print_console_report(results, cfg)